In [ ]:
# PDF CHATBOT

import os
os.environ["OPENAI_API_KEY"]=""

# Install Dependency
!pip install -U langchain-text-splitters
!pip install langchain-openai
!pip install -U langchain faiss-cpu

In [ ]:
# Import necessary libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Indexing Phase
# Step 1. Load PDF
loader = PyPDFLoader("path_to_file.pdf")
document = loader.load()

In [ ]:
# Step 2. Split Text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

chunks = text_splitter.split_documents(document)

In [ ]:
# Step 3. Create Vector Store
embedding = OpenAIEmbeddings()
vector_store = FAISS.from_documents(chunks, embedding)
vector_store.index_to_docstore_id

In [ ]:
# Retrival Phase

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

In [ ]:
# Augmented Generation

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant. Use the following retrieved context to answer the question.
    If you don't know the answer, say you don't know.
    Context: {context}
    Question: {question}
    """,
    input_variables=["context", "question"],
)

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough(),
})

In [ ]:
llm = ChatOpenAI(temperature=0.1, model="gpt-3.5-turbo")

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
response = main_chain.invoke("What is the main topic of the PDF?")